In [1]:
import cbbpy.mens_scraper as s
from datetime import datetime, timedelta
from concurrent.futures import ThreadPoolExecutor, as_completed
import pandas as pd

def retrieve_game(game_id: int):
    """
    Returns boxscore_df and info_df for a single game.
    """
    info_df, boxscore_df, _= s.get_game(game_id=game_id, pbp=False, info=True)
    return boxscore_df, info_df

def compute_possessions(fga, fta, oreb, to):
    """Estimate possessions for a team."""
    return fga - oreb + to + (0.44 * fta)

def scrape_data(days_ago=2):
    """
    Scrapes all games from a given day.
    Returns two lists: boxscores and info dataframes.
    """
    date = datetime.today() - timedelta(days=days_ago)
    game_ids = s.get_game_ids(date)

    if not game_ids:
        print(f"No games found on {date.strftime('%Y-%m-%d')}")
        return [], []

    print(f"Found {len(game_ids)} game(s) on {date.strftime('%Y-%m-%d')}")

    boxscores = []
    info_list = []

    with ThreadPoolExecutor(max_workers=5) as executor:
        future_to_gid = {executor.submit(retrieve_game, gid): gid for gid in game_ids}
        for i, future in enumerate(as_completed(future_to_gid), start=1):
            gid = future_to_gid[future]
            try:
                boxscore_df, info_df = future.result()
                boxscores.append(boxscore_df)
                info_list.append(info_df)
                print(f"Retrieved {gid} ({i}/{len(game_ids)})")
            except Exception as e:
                print(f"Failed to retrieve {gid}: {e}")

    print("All games scraped!")
    boxscores = pd.concat(boxscores, ignore_index=True)
    info_list = pd.concat(info_list, ignore_index=True)

    return boxscores, info_list

def build_matchup_table(boxscores, info_list):
    """
    Combine boxscore and info_df to create matchup table with home/away/neutral.
    """
    # 
    boxscores = boxscores[boxscores['player'] == 'TEAM']
    cols = ['game_id', 'team', 'pts', 'fga', 'fta', 'oreb', 'to']
    boxscores = boxscores[cols]

    # Each game has two rows — split into "team" and "opponent" by pairing them
    df1 = boxscores.iloc[::2].reset_index(drop=True)   # first team in each game
    df2 = boxscores.iloc[1::2].reset_index(drop=True)  # second team in each game

    # Join side by side
    merged = pd.concat([
        df1.rename(columns=lambda c: c if c == 'game_id' else f'team_{c}'),
        df2.rename(columns=lambda c: f'opp_{c}' if c != 'game_id' else c).drop(columns='game_id')
    ], axis=1)

    # Create inverse rows (swap team/opp)
    inverse = merged.rename(columns=lambda c: c.replace('team_', '__tmp__').replace('opp_', 'team_').replace('__tmp__', 'opp_'))

    # Stack original + inverse
    result = pd.concat([merged, inverse], ignore_index=True).sort_values('game_id').reset_index(drop=True)

    result = result.merge(info_list[['game_id', 'home_team', 'is_neutral', 'game_day']], on = 'game_id')
    result = result.rename(columns={'team_team': 'team'})
    
    result['neutral'] = result['is_neutral'] == 1
    result['home'] = (~result['neutral']) & (result['home_team'] == result['team'])
    result['away'] = (~result['neutral']) & (result['home_team'] != result['team'])

    result['possession_team'] = compute_possessions(result['team_fga'], result['team_fta'], result['team_oreb'], result['team_to'])
    result['possession_opp'] = compute_possessions(result['opp_fga'], result['opp_fta'], result['opp_oreb'], result['opp_to'])

    result['ppp_off_team'] = result['team_pts']/ result['possession_team']
    result['ppp_def_team'] = result['opp_pts'] / result['possession_opp']

    result['ppp_off_opp'] = result['opp_pts'] / result['possession_opp'] 
    result['ppp_def_opp'] = result['team_pts'] / result['possession_team']

    return result


# Example usage in a Jupyter notebook
boxscores, info_list = scrape_data(days_ago=2)
print(info_list)
matchup_df = build_matchup_table(boxscores, info_list)
matchup_df

Found 21 game(s) on 2026-02-27
Retrieved 401814590 (1/21)
Retrieved 401828448 (2/21)
Retrieved 401818335 (3/21)
Retrieved 401825544 (4/21)
Retrieved 401813395 (5/21)
Retrieved 401823856 (6/21)
Retrieved 401813432 (7/21)
Retrieved 401818546 (8/21)
Retrieved 401813418 (9/21)
Retrieved 401813371 (10/21)
Retrieved 401813318 (11/21)
Retrieved 401808700 (12/21)
Retrieved 401812623 (13/21)
Retrieved 401812596 (14/21)
Retrieved 401808694 (15/21)
Retrieved 401814586 (16/21)
Retrieved 401808697 (17/21)
Retrieved 401808695 (18/21)
Retrieved 401808699 (19/21)
Retrieved 401808696 (20/21)
Retrieved 401808698 (21/21)
All games scraped!
      game_id game_status                          home_team home_id  \
0   401814590       Final           Western Michigan Broncos    2711   
1   401828448       Final  George Washington Revolutionaries      45   
2   401818335       Final                    Cornell Big Red     172   
3   401825544       Final           Illinois Fighting Illini     356   
4   4018133

,game_id,team,team_pts,team_fga,team_fta,team_oreb,team_to,opp_team,opp_pts,opp_fga,...,game_day,neutral,home,away,possession_team,possession_opp,ppp_off_team,ppp_def_team,ppp_off_opp,ppp_def_opp
0,401808694,Georgia State Panthers,73,63,34,12,6,Old Dominion Monarchs,81,53,...,"February 27, 2026",False,True,False,71.96,73.20,1.014452,1.106557,1.106557,1.014452
1,401808694,Old Dominion Monarchs,81,53,30,6,13,Georgia State Panthers,73,63,...,"February 27, 2026",False,False,True,73.20,71.96,1.106557,1.014452,1.014452,1.106557
2,401808695,James Madison Dukes,68,58,21,11,7,Coastal Carolina Chanticleers,69,59,...,"February 27, 2026",False,True,False,63.24,62.60,1.075269,1.102236,1.102236,1.075269
3,401808695,Coastal Carolina Chanticleers,69,59,15,10,7,James Madison Dukes,68,58,...,"February 27, 2026",False,False,True,62.60,63.24,1.102236,1.075269,1.075269,1.102236
4,401808696,Marshall Thundering Herd,82,54,23,7,14,Georgia Southern Eagles,99,62,...,"February 27, 2026",False,True,False,71.12,70.56,1.152981,1.403061,1.403061,1.152981
5,401808696,Georgia Southern Eagles,99,62,24,10,8,Marshall Thundering Herd,82,54,...,"February 27, 2026",False,False,True,70.56,71.12,1.403061,1.152981,1.152981,1.403061
6,401808697,Texas State Bobcats,60,51,15,6,9,App State Mountaineers,57,57,...,"February 27, 2026",False,True,False,60.60,59.68,0.990099,0.955094,0.955094,0.990099
7,401808697,App State Mountaineers,57,57,22,20,13,Texas State Bobcats,60,51,...,"February 27, 2026",False,False,True,59.68,60.60,0.955094,0.990099,0.990099,0.955094
8,401808698,Louisiana Ragin' Cajuns,58,64,23,9,9,Arkansas State Red Wolves,81,62,...,"February 27, 2026",False,False,True,74.12,75.56,0.782515,1.071996,1.071996,0.782515
9,401808698,Arkansas State Red Wolves,81,62,24,9,12,Louisiana Ragin' Cajuns,58,64,...,"February 27, 2026",False,True,False,75.56,74.12,1.071996,0.782515,0.782515,1.071996


In [59]:
import pandas as pd
from datetime import datetime, timedelta

def build_full_season(start_date, end_date):

    all_games = []

    total_days = (end_date - start_date).days

    for days_ago in range(total_days, -1, -1):

        target_date = datetime.today() - timedelta(days=days_ago)
        date_str = target_date.strftime("%Y-%m-%d")

        try:

            boxscores, info_list = scrape_data(days_ago=days_ago)

            matchup_df = build_matchup_table(
                boxscores,
                info_list,
            )

            if not matchup_df.empty:
                all_games.append(matchup_df)

            print(f"Finished {date_str}")

        except Exception as e:

            print(f"Skipped {date_str}: {e}")

    full_season = pd.concat(all_games, ignore_index=True)

    full_season.to_csv('full_season.csv')

    return full_season

In [60]:
start = datetime(2025, 11, 3)
end   = datetime.today()

season_df = build_full_season(start, end)

Found 169 game(s) on 2025-11-03
Retrieved 401812785 (1/169)
Retrieved 401826885 (2/169)
Retrieved 401826083 (3/169)
Retrieved 401824809 (4/169)
Retrieved 401820577 (5/169)
Retrieved 401819834 (6/169)
Retrieved 401813756 (7/169)
Retrieved 401826784 (8/169)
Retrieved 401812260 (9/169)
Retrieved 401817239 (10/169)
Retrieved 401812282 (11/169)
Retrieved 401827170 (12/169)
Retrieved 401823483 (13/169)
Retrieved 401823474 (14/169)
Retrieved 401819882 (15/169)
Retrieved 401824031 (16/169)
Retrieved 401827219 (17/169)
Retrieved 401826911 (18/169)
Retrieved 401823449 (19/169)
Retrieved 401817262 (20/169)
Retrieved 401830655 (21/169)
Retrieved 401817194 (22/169)
Retrieved 401828511 (23/169)
Retrieved 401827039 (24/169)
Retrieved 401826859 (25/169)
Retrieved 401829447 (26/169)
Retrieved 401825586 (27/169)
Retrieved 401824943 (28/169)
Retrieved 401827511 (29/169)
Retrieved 401823513 (30/169)
Retrieved 401827734 (31/169)
Retrieved 401827274 (32/169)
Retrieved 401824867 (33/169)
Retrieved 401823425 

In [63]:
print(season_df['team'].unique())

['New Orleans Privateers' 'TCU Horned Frogs' 'Ole Miss Rebels'
 'SE Louisiana Lions' 'Delaware Blue Hens' 'Bucknell Bison'
 'Miami Hurricanes' 'Jacksonville Dolphins' 'Utah Tech Trailblazers'
 'South Dakota Coyotes' 'Air Force Falcons' 'Belmont Bruins'
 'North Dakota Fighting Hawks' 'Alabama Crimson Tide'
 'Fairleigh Dickinson Knights' 'Iowa State Cyclones'
 'Long Island University Sharks' 'Notre Dame Fighting Irish'
 'Oklahoma Sooners' 'Saint Francis Red Flash' 'Virginia Cavaliers'
 'Rider Broncs' 'Vermont State - Johnson Badgers'
 'Central Connecticut Blue Devils' 'Holy Cross Crusaders'
 'Providence Friars' 'UConn Huskies' 'New Haven Chargers'
 'New Hampshire Wildcats' 'Clemson Tigers' 'Georgetown Hoyas'
 'Morgan State Bears' 'Marist Red Foxes' 'Xavier Musketeers'
 'Canisius Golden Griffins' 'Dayton Flyers' 'Niagara Purple Eagles'
 'Duquesne Dukes' 'Eastern Washington Eagles' 'UCLA Bruins'
 'Maine Black Bears' 'George Washington Revolutionaries'
 'Loyola Chicago Ramblers' 'Cleveland 